# 02 - Pré-processamento e Limpeza dos Dados

Aplica as decisões tomadas na EDA (`01_eda`) para gerar um dataset limpo e confiável:

1. Remoção de duplicatas
2. Remoção de outliers impossíveis (`person_age` > 100, `person_emp_length` > 60)
3. Imputação de valores ausentes (`loan_int_rate`, `person_emp_length`) pela mediana

O resultado é salvo em `data/processed/credit_clean.csv` — o **"contrato de dados"** que
desbloqueia o dashboard (Igor) e a etapa seguinte de engenharia de features + modelagem.

> A **engenharia de features** (ratios, binning, encoding, scaling) será feita na etapa seguinte,
> dentro do pipeline de modelagem, para evitar vazamento de dados (fit só no treino).

## 1. Setup e carregamento dos dados brutos

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Raiz do projeto (um nível acima de notebooks/), para importar o pacote src
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import load_raw_data, clean_data, save_processed_data

df_raw = load_raw_data(PROJECT_ROOT / "data/raw/credit_risk_dataset.csv")
print(f"Dataset bruto: {df_raw.shape[0]} linhas x {df_raw.shape[1]} colunas")

Dataset bruto: 32581 linhas x 12 colunas


## 2. Diagnóstico antes da limpeza

Confirmamos os problemas levantados na EDA.

In [2]:
print("Duplicatas:", df_raw.duplicated().sum())
print("\nValores ausentes:")
print(df_raw.isna().sum()[df_raw.isna().sum() > 0])
print("\nOutliers:")
print("  person_age > 100:", (df_raw["person_age"] > 100).sum())
print("  person_emp_length > 60:", (df_raw["person_emp_length"] > 60).sum())

Duplicatas: 165

Valores ausentes:
person_emp_length     895
loan_int_rate        3116
dtype: int64

Outliers:
  person_age > 100: 5
  person_emp_length > 60: 2


## 3. Limpeza

Aplicamos `clean_data` (definida em `src/data.py`), que centraliza as três etapas de limpeza.

In [3]:
df_clean = clean_data(df_raw)
print(f"Antes:  {df_raw.shape[0]} linhas")
print(f"Depois: {df_clean.shape[0]} linhas")
print(f"Removidas: {df_raw.shape[0] - df_clean.shape[0]} linhas "
      f"({(df_raw.shape[0] - df_clean.shape[0]) / df_raw.shape[0]:.2%})")

Antes:  32581 linhas
Depois: 32409 linhas
Removidas: 172 linhas (0.53%)


## 4. Validação depois da limpeza

Garantimos que não restaram ausentes, duplicatas nem outliers — e que a distribuição do alvo se manteve.

In [4]:
assert df_clean.isna().sum().sum() == 0, "Ainda há valores ausentes!"
assert df_clean.duplicated().sum() == 0, "Ainda há duplicatas!"
assert df_clean["person_age"].max() <= 100, "Ainda há idade > 100!"
assert df_clean["person_emp_length"].max() <= 60, "Ainda há emp_length > 60!"

print("OK - sem ausentes, sem duplicatas, sem outliers impossíveis.")
print("\nValores ausentes restantes:", df_clean.isna().sum().sum())
print("Idade máxima:", df_clean["person_age"].max())
print("Tempo de emprego máximo:", df_clean["person_emp_length"].max())

OK - sem ausentes, sem duplicatas, sem outliers impossíveis.

Valores ausentes restantes: 0
Idade máxima: 94
Tempo de emprego máximo: 41.0


In [5]:
# A proporção do alvo deve permanecer praticamente igual (não enviesamos a base)
print("Proporção do alvo ANTES:")
print(df_raw["loan_status"].value_counts(normalize=True).round(4))
print("\nProporção do alvo DEPOIS:")
print(df_clean["loan_status"].value_counts(normalize=True).round(4))

Proporção do alvo ANTES:
loan_status
0    0.7818
1    0.2182
Name: proportion, dtype: float64

Proporção do alvo DEPOIS:
loan_status
0    0.7813
1    0.2187
Name: proportion, dtype: float64


In [6]:
df_clean.describe().T

,count,mean,std,min,25%,50%,75%,max
person_age,32409.0,27.730754,6.210445,20.00,23.00,26.00,30.00,94.00
person_income,32409.0,65894.277053,52517.869973,4000.00,38500.00,55000.00,79200.00,2039784.00
person_emp_length,32409.0,4.761424,3.983757,0.00,2.00,4.00,7.00,41.00
loan_amnt,32409.0,9592.486655,6320.885127,500.00,5000.00,8000.00,12250.00,35000.00
loan_int_rate,32409.0,11.014512,3.083104,5.42,8.49,10.99,13.11,23.22
loan_status,32409.0,0.218705,0.413374,0.00,0.00,0.00,0.00,1.00
loan_percent_income,32409.0,0.170248,0.106785,0.00,0.09,0.15,0.23,0.83
cb_person_cred_hist_length,32409.0,5.811194,4.057899,2.00,3.00,4.00,8.00,30.00


## 5. Salvar o dataset limpo (contrato de dados)

Salvo em `data/processed/credit_clean.csv`. Esse arquivo é a base para o dashboard e a modelagem.

In [7]:
saida = PROJECT_ROOT / "data/processed/credit_clean.csv"
save_processed_data(df_clean, saida)
print(f"Salvo em: {saida}")
print(f"Shape final: {df_clean.shape}")

Salvo em: /home/marco/Área de trabalho/credit-risk-analysis-ds/data/processed/credit_clean.csv
Shape final: (32409, 12)


## Próximos passos

- **Engenharia de features** (na etapa de modelagem, `03_modeling`): ratios `loan_to_income_ratio`,
  `loan_to_emp_length_ratio`, `int_rate_to_loan_amt_ratio`; faixas de idade/renda; encoding das
  categóricas; padronização das numéricas — tudo dentro de um `Pipeline` ajustado **só no treino**.
- **Modelagem**: comparar Regressão Logística (baseline), Random Forest e Gradient Boosting
  (XGBoost/LightGBM), tratando o desbalanceamento, com métricas ROC-AUC, recall, F1 e matriz de confusão.